## Example 4.1 : loss of significance in float64 calculation

### What you will learn

In this notebook, we:
- see how float64 calculation is done by explicitly writing down the 64 bits manipulation
- see how loss of significance occurs 
- The example is the following:

In [1]:
from math import sqrt

x = 1.0
y = 1.0 + (1e-14) *sqrt(2)
print((1e14) *(y-x))
print(sqrt(2))

1.4210854715202004
1.4142135623730951


We see that the result of (1e14) *(y-x) differs significantly from the true value, $\sqrt{2}$.  
**In this note, we see how this result (1.4210854715202004) arises, by explicitly investigating the 64 bit calculation following the float64 calculation rule.**  
This is done step by step:
 - Calculation of (1e-14) *sqrt(2)
 - Calculation of 1.0 + (1e-14) *sqrt(2)
 - Calculation of [1.0 + (1e-14) *sqrt(2)] - 1.0
 - Calculation of (1e14)*[[1.0 + (1e-14) *sqrt(2)] - 1.0]

## Task
To repeat our task:  
Perform the float64 calculation of  (1e14)*[[1.0 + (1e-14) *sqrt(2)] - 1.0] **explicitly**.  

## Theory
Before tackling on our task, let us review how float64 is represented, and how they are added or multiplied (we do not treat division in this note).    
cf.  
[https://en.wikipedia.org/wiki/Double-precision_floating-point_format](https://en.wikipedia.org/wiki/Double-precision_floating-point_format)  
[https://www.rfwireless-world.com/tutorials/ieee-754-floating-point-arithmetic](https://www.rfwireless-world.com/tutorials/ieee-754-floating-point-arithmetic)


### Representation of float64
The 64 bit is decomposed as 1 bit sign part ($s$), 11 bit exponent part ($E$), and 52 bit fraction part ($f$ with $0<f<1$), which make a real number by:
$$
r = (-1)^s (1+f) 2^{E-1023}
$$
(we do not consider denormalized numbers here).  In this note, we consider only positive numbers, so we omit the sign part $s$ in the following discussion for simplicity. 

Example: Take $3.3$ for example.  $3.3=1.65\times 2$, and $0.65$ is represented by binary as 
$$
0.\underbrace{1010011001100110011001100110011001100110011001100110}_{52\text{bits}}011\cdots_2.
$$
Rounding to 52 bits gives (cf. "Rounding" section below)
$$
0.\underbrace{1010011001100110011001100110011001100110011001100110}_{52\text{bits}}{}_2.
$$
After all, $3.3$ is represented by the exponent part $E=10000000000_2$ and $f=1010011001100110011001100110011001100110011001100110_2$.

### Multiplication
Consider multiplication of two real numbers, $r_3=r_1r_2$.  Below, we represent each number as $r_i = (1+f_i) 2^{E_i-1023}$.  
Since
$$
r_1r_2 = (1+f_1)(1+f_2) 2^{E_1+E_2-2046},
$$
then
$$
1+f_3 = (1+f_1)(1+f_2), ~~~ E_3 = E_1 + E_2 - 1023.
$$

### Addition (and similarly for subtraction)
Consier $r_3=r_1+r_2$.  Assume that the absolute value of $r_1$ is larger than that of $r_2$.  We first adjust the representation of $r_2$ as $r_2 = \frac{1+f_2}{2^d} 2^{E_2+d-1023}$ so that $E_2+d=E_1$.  Then we simply perform addition:  
$$
r_3 = \left(1+f_1 + \frac{1+f_2}{2^d} \right)2^{E_1-1023}.
$$


### Normalization 
In some cases, after computations (multiplication or addition etc.), the fractional part $f$ violates the condition $0\leq f<1$.  In those cases, we adjust the exponent part so that $0\leq f<1$ holds.  

Example:  Consider multiplication of $1.4*1.8$.  After multiplication, the fractional part becomes $1+f=1.4*1.8=2.52$ and $E=1023+1023-1023=1023$.  Normalize $2.52$ by halving $2.52/2=1.26$ and instead increasing $E$ to $1023+1=1024$.


### Rounding 
We have to round fractional part $f$ so that it is represented by 52-bit binary.  
In most cases, the rounding rule is simple: round down if the 53rd bit is 0, and round up if it is 1.  
The only case to be careful is the case where the 53rd bit is 1 and all the bits after the 53rd bit is 0 ("tie case").  In this case, round up/down the number so that the 52nd bit becomes 0 ("round half to even"). 

## Task execution

Now let us execute the calculation of (1e14)*[[1.0 + (1e-14) *sqrt(2)] - 1.0] step by step, following the prescription (thoery) described above.

### Helper functions 1

Below are functions to perform conversion between decimal representation and its float64 binary representation.

In [2]:
import numpy as np
from functools import partial

def decompose_float64(d: float) -> tuple[str, str, str]:  # given a 64bit double, this function returns its sign, exponent, fraction part as strings
    x = np.float64(d)
    x_int64 = x.view(np.int64)  # reinterpret the bits of x as those of int64 and returns the corresponding int64 number 
    binary_str = str(f"{x_int64:064b}")  # returns the string of binary representation of x_int64
    sign_str = binary_str[0]
    exponent_str = binary_str[1:12]
    fraction_str = binary_str[12:]
    return sign_str, exponent_str, fraction_str

def compose_float64(sign_str: str, exponent_str: str, fraction_str: str) -> float:  # this is the "inverse" of decompose_float64
    assert len(sign_str)==1 and len(exponent_str)==11 and len(fraction_str)==52
    assert set(sign_str)<= {'0', '1'} and set(exponent_str)<= {'0', '1'} and set(fraction_str)<= {'0', '1'}

    binary_str = sign_str + exponent_str + fraction_str
    x = int(binary_str, 2)   # interpret binary_str as binary representation of an integer and returns that integer
    x_int64 = np.int64(x)
    x_float64 = x_int64.view(np.float64)  # reinterpret the bits of x_int64 as those of float64 and returns the corresponding float64 number

    return x_float64.item()

In [3]:
# check
print(decompose_float64(3.3))
print(compose_float64(*decompose_float64(3.3)))

('0', '10000000000', '1010011001100110011001100110011001100110011001100110')
3.3


In [29]:
def multiply_binary_strings(a: str, b: str) -> str:
    """
    a, b: assumes '1.xxx...' where xxx... is a 52-bit binary
    returns a string representing the result of a*b 
    In the result, '_' is inserted as a mark to the right of the 52nd bit after the binary point 
    """
    assert a.startswith('1.') and b.startswith('1.')
    assert len(a) == len(b) == 54   # '1.' + 52 bits 

    frac_len = 52

    # remove period
    a_int = a.replace('.', '')
    b_int = b.replace('.', '')

    # reverse（process from lower bits）
    a_rev = a_int[::-1]
    b_rev = b_int[::-1]

    # a variable to store the result
    result = [0] * (len(a_int) + len(b_int))

    # --- calculation ---
    for i, bit_b in enumerate(b_rev):
        if bit_b == '1':
            for j, bit_a in enumerate(a_rev):
                if bit_a == '1':
                    result[i + j] += 1

    # --- carry over ---
    carry = 0
    for i in range(len(result)):
        total = result[i] + carry
        result[i] = total % 2
        carry = total // 2

    # convert to a string
    result_str = ''.join(map(str, result[::-1]))

    # restore the binary point
    total_frac = frac_len * 2
    integer_part = result_str[:-total_frac].lstrip('0')
    fraction_part = result_str[-total_frac:]

    return integer_part + '.' + fraction_part[:52] + '_' + fraction_part[52:]

### Calculation of (1e-14) *sqrt(2)

In [16]:
# First we get float64 binary representation of the operands
s_a, e_a, f_a = decompose_float64(1e-14)
print("1e-14 is decomposed as:  ", s_a, e_a, f_a)
s_b, e_b, f_b = decompose_float64(sqrt(2))
print("sqrt(2) is decomposed as:", s_b, e_b, f_b)

# Then multiply (1+f) parts
print("the result of (1+f_a)(1+f_b):")
multiply_binary_strings('1.'+f_a, '1.'+f_b)

1e-14 is decomposed as:   0 01111010000 0110100001001001101110000110101000010010101110011011
sqrt(2) is decomposed as: 0 01111111111 0110101000001001111001100110011111110011101111001101
the result of (1+f_a)(1+f_b):


'1.1111110110000110001011011010001000000010100101010110_1111110100111001000001011000000000011010010000011111'

The result is already in the form of '1.xxx...', so we do not need normalization.  By rounding up, the resultant fraction part is $\texttt{1111110110000110001011011010001000000010100101010111}$.  
As for exponentail part, the result is $\texttt{01111010000} + \texttt{01111111111} - \texttt{01111111111} = \texttt{01111010000}$.  
In all, (1e-14) *sqrt(2) results in float 64 with 
$s=\texttt{0}, E=\texttt{01111010000}, f=\texttt{1111110110000110001011011010001000000010100101010111}$.  

### Calculation of 1.0 + (1e-14) *sqrt(2)

In [17]:
decompose_float64(1.0)

('0', '01111111111', '0000000000000000000000000000000000000000000000000000')

The difference in the exponential parts between 1.0 and (1e-14) *sqrt(2) is $\texttt{01111111111} - \texttt{01111010000} = \texttt{101111} = {47}_{10}$.  So,  we need to shift the 1+f part of (1e-14) *sqrt(2) by 47 times:
$$
\begin{aligned}
1+f=&\texttt{1.1111110110000110001011011010001000000010100101010111}  \\
&\downarrow (\text{47 bit right-shift})\\
    &\texttt{0.000000000000000000000000000000000000000000000011111110110000110001011011010001000000010100101010111} \\
&\downarrow (\text{rounding})\\
    &\texttt{0.0000000000000000000000000000000000000000000001000000}
\end{aligned}
$$
Summing this to the 1+f part of 1.0 gives 
$$
\texttt{1.0000000000000000000000000000000000000000000001000000}
$$
In all, 1.0 + (1e-14) *sqrt(2) results in float64 representation as 
$s=\texttt{0}, E=\texttt{01111111111}, f=\texttt{0000000000000000000000000000000000000000000001000000}$. 

### Calculation of [1.0 + (1e-14) *sqrt(2)] - 1.0

As for the 1+f part, subtracting 
$$
1.0000000000000000000000000000000000000000000000000000
$$
from 
$$
1.0000000000000000000000000000000000000000000001000000
$$
results in 
$$
\underbrace{0.000000000000000000000000000000000000000000000}_{46\text{bits}}\underbrace{1000000}_{7\text{bits}}
$$
Perform normalization by shifting 46 times!
$$
\underbrace{1.000000}_{\text{significant}}\underbrace{0000000000000000000000000000000000000000000000}_{\text{not significant}}
$$
<span style="color:#b22222; font-weight:bold">Note that only the first 7 bits are significant, and the remaining 46 bits are not (i.e., do not have any information).  Thus, loss of significance occurs.</span>  
As for the exponential part, $01111111111_2 - 46_{10} = 01111010001_2$.  
In all, [1.0 + (1e-14) *sqrt(2)] - 1.0 results in float64 representation as 
$s=\texttt{0}, E=\texttt{01111010001}, f=\texttt{0000000000000000000000000000000000000000000000000000}$.  

### Calculation of ([1.0 + (1e-14) *sqrt(2)] - 1.0) * 1e14

In [22]:
decompose_float64(1e14)

('0', '10000101101', '0110101111001100010000011110100100000000000000000000')

As for the exponential part, $01111010001_2+10000101101_2-1023_{10} = 01111111111_2$.  The fractional part remains the same. after multiplied by 1.0.  
In all,([1.0 + (1e-14) *sqrt(2)] - 1.0) * 1e14 results in float64 representation as 
$s=\texttt{0}, E=\texttt{01111111111}, f=\texttt{0110101111001100010000011110100100000000000000000000}$.  

Finally, let us convert this float64 representation to decimal number:

In [24]:
compose_float64(sign_str='0', exponent_str='01111111111', fraction_str='0110101111001100010000011110100100000000000000000000')

1.4210854715202004

In [28]:
(1.0 + (1e-14) *sqrt(2) - 1.0) * 1e14

1.4210854715202004

textttをなくす、exponentやfractionの計算をもっと見出し風にして分ける

In [ ]:
print(parse_double(1e-14))
print(parse_double(sqrt(2)))

A = 01111010000____0110100001001001101110000110101000010010101110011011  
B = 01111111111____0110101000001001111001100110011111110011101111001101  
Prepend "1." to the fraction part:  
A = 01111010000__1.0110100001001001101110000110101000010010101110011011  
B = 01111111111__1.0110101000001001111001100110011111110011101111001101  
Add the biased exponent:  
01111010000 + 01111111111 - 01111111111 = 01111010000  
Multiply the 1+f parts:  
1.0110100001001001101110000110101000010010101110011011 * 1.0110101000001001111001100110011111110011101111001101 = 1.111111011000011000101101101000100000001010010101011011 (truncated to 55 digits)  
Then C=A*B is:  
C = 01111010000____1.111111011000011000101101101000100000001010010101011011  
Normalize: (unnecessary)  
Round:  
C = 01111010000____1.1111110110000110001011011010001000000010100101010111  

The intermediate result is as follows:

In [ ]:
make_double("0", "01111010000", "1111110110000110001011011010001000000010100101010111")

### Calculation of 1.0 + (1e-14) *sqrt(2)

In [ ]:
print(parse_double(1.0))

A = 01111010000____1111110110000110001011011010001000000010100101010111  (the previous result)  
B = 01111111111____0000000000000000000000000000000000000000000000000000  
Prepend "1." to the fraction part, the result of which we call "1+f part":  
A = 01111010000__1.1111110110000110001011011010001000000010100101010111  
B = 01111111111__1.0000000000000000000000000000000000000000000000000000  
Make the exponent of A as the same as that of B, and shift the 1+f part to the right:  
Since 01111111111 - 01111010000 = 101111 = (47)_10, we shift the 1+f part of A by 47 times!!  
A = 01111111111__0.0000000000000000000000000000000000000000000000111111101100 (truncated)   
B = 01111111111__1.0000000000000000000000000000000000000000000000000000  
Add the 1+f parts, and call the result C:   
C = 01111111111__1.0000000000000000000000000000000000000000000000111111101100  
Round :  
C = 01111111111__1.0000000000000000000000000000000000000000000001000000  
 
The intermediate result is as follows:

In [ ]:
make_double("0", "01111111111", "0000000000000000000000000000000000000000000001000000")

### Calculation of [1.0 + (1e-14) *sqrt(2)] - 1.0

In [ ]:
print(parse_double(1.0))

A = 01111111111____1.0000000000000000000000000000000000000000000001000000  (previous result)  
B = 01111111111____1.0000000000000000000000000000000000000000000000000000
Subtract the 1+f part of B from that of A:  
C = 01111111111____0.0000000000000000000000000000000000000000000001000000  
Normalize :  
We should shift the 1+f part of C to the left 46 times !!  Since (46)_10 = (101110)_2, the exponent part becomes 01111111111 - 101110 = 01111010001  
C = 01111010001____1.0000000000000000000000000000000000000000000000000000    

Thus we have seen that the combination of "large number of right-shift" + "truncation" + "large number of left-shit" yields large numerical error.  
The intermediate result is:

In [ ]:
make_double("0", "01111010001", "0000000000000000000000000000000000000000000000000000")

### Calculation of ([1.0 + (1e-14) *sqrt(2)] - 1.0) * 1e14
The result might be obvious, but we perform calculation anyway.

In [ ]:
print(parse_double(1e14))

A = 01111010001____1.0000000000000000000000000000000000000000000000000000  (previous result)  
B = 10000101101____1.0110101111001100010000011110100100000000000000000000  
Add the biased exponent:  
01111010001 + 10000101101 - 01111111111 = 01111111111  
Multiply the 1+f parts:  
the result is clearly 1.0110101111001100010000011110100100000000000000000000  
Then C=A*B is:  
C = 01111111111____1.0110101111001100010000011110100100000000000000000000  
Normalize: (unnecessary) 
Round: (unnecessary)  

Thus the final result is:

In [ ]:
make_double("0", "01111111111", "0110101111001100010000011110100100000000000000000000")

This coincides with the expected value : 

In [ ]:
((1.0 + (1e-14) *sqrt(2)) - 1.0) * 1e14

when we take another look : 

In [ ]:
a = sqrt(2)
print("step 1: ", f"{a:.110f}")
a = 1e-14 * a
print("step 2: ", f"{a:.110f}")
a = 1.0 + a
print("step 3: ", f"{a:.110f}")
a = a - 1.0
print("step 4: ", f"{a:.110f}")
a = 1e14 * a
print("step 5: ", f"{a:.110f}")